In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from scipy.ndimage import uniform_filter
import h5py
import numpy as np
import pandas as pd
import os
import glob

In [2]:
# Filepaths
ICA_Type = 'Spatial'
n_components = 5
StorePath = '/work/xfc/vol7/user_cache/benirela/Other/Pedro_TS/'
EndPath = '/TS_GEOCml2clipmask/cum_filt.h5'
Volcanoes = ['Wolf','Alcedo','Cerro_Azul_Galapagos','Darwin','Fernandina','Sierra_Negra','Nabro','Asavyo','Dabbahu']
Frames = ['106A','106A','106A','128D','128D','128D','014A','014A','014A']

In [ ]:
# Load timeseries
for i, (Volc, Frame) in enumerate(zip(Volcanoes,Frames)):
    #TS_File = StorePath + '/' + Volc + '/' + Frame + EndPath
    TS_File = glob.glob(os.path.join(StorePath, Volc, Frame, 'TS_GEO*', 'cum_filt.h5'))
    TS_File = TS_File[0]

    # Open timeseries file
    with h5py.File(TS_File, "r") as f:
        ImDates = f["/imdates"][:]
        LOS = f["/cum"][:]
        cLat = f["/corner_lat"][()]
        cLon = f["/corner_lon"][()]
        postLat = f["/post_lat"][()]
        postLon = f["/post_lon"][()]

    # Convert dates
    DatesDT = pd.to_datetime(ImDates.astype(str), format="%Y%m%d")
    days_ = (DatesDT - DatesDT[0]).days.to_numpy()

    LOS = LOS.transpose((1, 2, 0))  # Transpose to (lat, lon, time)

    # Find extent
    endLat = (LOS.shape[0] - 1) * postLat + cLat
    endLon = (LOS.shape[1] - 1) * postLon + cLon

    # Create lat and lon postings and grids
    lat = np.arange(cLat, endLat + postLat/2, postLat)
    lon = np.arange(cLon, endLon + postLon/2, postLon)
    Lon, Lat = np.meshgrid(lon, lat)

    # Covert from mm to m
    LOS = LOS / 1000.0

    # Find maximum absolute displacement
    CumDisp = LOS[:, :, -1]
    MaxDisp = np.nanmax(abs(CumDisp))

    print(str(i+1) + ' of ' + str(len(Frames)) + ' timeseries loaded')

    # # Check in NaNs are the same between all temporal slices
    # nan_mask = np.isnan(LOS)
    # same_nan_pattern = np.all(nan_mask == nan_mask[:, :, [0]])

    # Get into format for ICA (Shape = num. samples * num. features)
    numcols = LOS.shape[1]
    numrows = LOS.shape[0]
    num_pix = numrows * numcols
    num_time = LOS.shape[2]
    X = LOS.transpose(2, 0, 1).reshape(num_time, num_pix)
    # Samples = num. timeseries
    # Features = num. pixels
    # Spatially independent sources and unconstrained time courses are recovered

    # Find pixels that are valid for every time step
    valid_pixels = ~np.isnan(X).any(axis=0)
    X = X[:, valid_pixels]

    if ICA_Type == 'Temporal':
        # Samples = num. pixels
        # Features = num. timeseries
        # Independent time courses and unconstrained spatial sources are recovered
        X = X.T

    
    # Test FastICA
    # Do ICA
    ica = FastICA(n_components, whiten="unit-variance")
    S_Estimated = ica.fit_transform(X) # Recovered signals
    A_estimated = ica.mixing_ # Mixing matrix

    Reconstruct_Orig = np.full((num_pix, num_time), np.nan)
    X_rec = ica.inverse_transform(S_Estimated)
    # Reconstruct signals 
    if ICA_Type == "Spatial":
        Reconstruct_Orig[valid_pixels, :] = X_rec.T
    elif ICA_Type == "Temporal":
        Reconstruct_Orig[:, valid_pixels] = X_rec

    Reconstruct_Orig = Reconstruct_Orig.reshape(numrows, numcols, num_time)
    # Reconstruct signals
    # if ICA_Type == "Temporal":
    #     Reconstruct_Orig = Reconstruct_Orig.T
    #     Reconstruct_Orig[:, valid_pixels] = A_estimated @ S_Estimated.T + ica.mean_[:, np.newaxis]
    #     Reconstruct_Orig = Reconstruct_Orig.T
    # else:
    #     Reconstruct_Orig[valid_pixels, :] = A_estimated @ S_Estimated.T + ica.mean_[:, None]

    # Reconstruct_Orig = Reconstruct_Orig.reshape(numrows,numcols,num_time)

    ICA_Reconstructed = np.zeros((X.shape[1],X.shape[0],n_components))
    fig, axs = plt.subplots(n_components+1,2,figsize=(15, 25))
    for i in range(n_components):
        ICA_Reconstructed[:,:,i] = A_estimated[:,i][:, np.newaxis] * S_Estimated[:,i][:, np.newaxis].T

        # Recompute into datacube shape
        datacube_comp = np.full((num_pix, num_time), np.nan)
        if ICA_Type == "Temporal":
            datacube_comp = datacube_comp.T
            datacube_comp[:, valid_pixels] = ICA_Reconstructed[:,:,i]
            datacube_comp = datacube_comp.T
        else:
            datacube_comp[valid_pixels, :] = ICA_Reconstructed[:,:,i]
        datacube_comp = datacube_comp.reshape(numrows,numcols,num_time)
        datacube_comp = datacube_comp - datacube_comp[:, :, 0][:, :, np.newaxis] # Start from zero

        MaxD = np.nanmax(np.abs(datacube_comp[:,:,-1]))
        implot = axs[i,0].imshow(datacube_comp[:,:,-1],aspect='1',vmin=-MaxD, vmax=MaxD)
        # mean = uniform_filter(np.abs(datacube_comp[:,:,-1]), size=10, mode='constant')
        # MaxR, MaxC = np.unravel_index(np.nanargmax(mean), mean.shape)

        MaxIdx = np.nanargmax(np.abs(datacube_comp[:,:,-1]))
        MaxR, MaxC = np.unravel_index(MaxIdx,datacube_comp[:,:,-1].shape)

        axs[i,0].scatter(MaxC,MaxR,c='r')

        cbar = plt.colorbar(implot)
        cbar.set_label("LOS Displacement (m)")

        # Plot time-courses for eact IC
        axs[i,1].plot(DatesDT,datacube_comp[MaxR, MaxC, :])

        if i+1==n_components:
            # Plot original data and residual (removed data during PCA)
            MaxD = np.nanmax(np.abs(CumDisp))
            implot = axs[i+1,0].imshow(CumDisp,aspect='1',vmin=-MaxD, vmax=MaxD)
            cbar = plt.colorbar(implot)
            cbar.set_label("LOS Displacement (m)")
            axs[i+1,0].set_title('Original timeseries')

            Residual = CumDisp - Reconstruct_Orig[:,:,-1]
            implot = axs[i+1,1].imshow(datacube_comp[:,:,-1],aspect='1')
            cbar = plt.colorbar(implot)
            cbar.set_label("LOS Displacement (m)")
            axs[i+1,1].set_title('Removed by PCA')
    
    plt.suptitle(Volc + "-" + Frame + ": " + ICA_Type + " ICA")
    fig.savefig(Volc + "_" + Frame + "_" + ICA_Type + "_ICA.png")

1 of 9 timeseries loaded
2 of 9 timeseries loaded
3 of 9 timeseries loaded


/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


4 of 9 timeseries loaded
5 of 9 timeseries loaded
6 of 9 timeseries loaded
7 of 9 timeseries loaded
8 of 9 timeseries loaded
9 of 9 timeseries loaded
